In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {

    # ---------------- Dataset ----------------
    "auto_download_results": True,
    "results_file_name": "Lithuanian_sentiment_analysis_dataset_results.csv",
    "dataset_path": "https://raw.githubusercontent.com/MaheshKumarPratihar/VGTU_thesis/main/datasets/lithuanian-sentiment-analysis.parquet",
    "text_column": "text",
    "label_column": "sentiment",

    # ---------------- PCA ----------------
    # ---------------- PCA ----------------
    # "use_pca": False,
    # "pca_components": 100,

    # "use_pca": True,
    # "pca_components": 50,

    "use_pca": True,
    "pca_components": 300,

    # ---------------- SOM ----------------
    # "som_grids": [(5, 5), (8, 8), [10, 10]],
    # "sigmas": [0.5, 1.0, 1.5, 2.0],
    # "learning_rates": [0.1, 0.3, 0.5],
    # "som_iterations": 1000,
    # "random_seed": 42, # [42, 7, 21],

    "som_grids": [(5, 5)],
    "sigmas": [2.0],
    "learning_rates": [0.1],
    "som_iterations": 1000,
    "random_seed": 42, # [42, 7, 21],

    # ---------------- Embedding Models ----------------
    "embedding_models": [
        "Word2Vec_avg",
        "Doc2Vec",
        "MiniLM",
        "MPNet",
        "DistilBERT_STS"
    ]
}

In [ ]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================
# Run this separately in Jupyter/Colab:
!pip install numpy pandas scikit-learn minisom \
sentence-transformers gensim matplotlib
# ============================================================
# IMPORTS
# ============================================================
import gc
import re
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import shutil
from minisom import MiniSom
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)
from sklearn.metrics.cluster import contingency_matrix
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sentence_transformers import SentenceTransformer
from collections import Counter, defaultdict

folder_path = "/content/" + CONFIG["results_file_name"].replace(".csv", "")

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
from IPython.display import clear_output
clear_output()

In [ ]:
# ============================================================
# SOM VISUALIZATION FUNCTIONS
# ============================================================

def create_output_dir(dataset_name):
    folder = f"som_plots_{dataset_name}"
    os.makedirs(folder, exist_ok=True)
    return folder


def safe_filename(name):
    return (
        str(name)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )


def plot_umatrix_with_class_count_markers(
    som,
    data,
    true_labels,
    label_encoder,
    model_name,
    som_x,
    som_y,
    sigma,
    lr,
    output_dir
):
    """
    U-Matrix with one marker per class per neuron.

    Background:
        U-Matrix distance map.

    Marker:
        One marker represents one true class inside one SOM neuron.

    Number:
        The number inside/near the marker shows how many samples
        of that class were mapped to that neuron.

    This avoids overcrowding when many samples map to the same neuron.
    """

    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from collections import defaultdict, Counter

    grid_size = f"{som_x}x{som_y}"

    plt.figure(figsize=(11, 8))

    # ---------------- U-Matrix background ----------------
    distance_map = som.distance_map().T

    plt.pcolor(
        distance_map,
        cmap="bone_r"
    )

    plt.colorbar(
        label="Distance between neurons"
    )

    # ---------------- Marker settings ----------------
    markers = [
        "o", "s", "^", "D", "v",
        "P", "X", "*", "<", ">"
    ]

    colors = [
        "red", "green", "blue", "orange", "purple",
        "brown", "pink", "cyan", "black", "gray"
    ]

    unique_labels = np.unique(true_labels)

    # ---------------- Count classes per neuron ----------------
    neuron_class_counts = defaultdict(Counter)

    for sample, label in zip(data, true_labels):
        winner = som.winner(sample)
        neuron_class_counts[winner][int(label)] += 1

    # Small offsets so multiple class markers in same neuron do not overlap
    offsets = [
        (-0.22, -0.22),
        (0.22, -0.22),
        (-0.22, 0.22),
        (0.22, 0.22),
        (0.00, 0.00),
        (-0.30, 0.00),
        (0.30, 0.00),
        (0.00, -0.30),
        (0.00, 0.30),
        (0.30, 0.30)
    ]

    # ---------------- Plot one marker per class per neuron ----------------
    for neuron, class_counts in neuron_class_counts.items():

        sorted_class_counts = class_counts.most_common()

        for idx, (label, count) in enumerate(sorted_class_counts):

            marker = markers[label % len(markers)]
            color = colors[label % len(colors)]

            offset_x, offset_y = offsets[idx % len(offsets)]

            x_pos = neuron[0] + 0.5 + offset_x
            y_pos = neuron[1] + 0.5 + offset_y

            plt.plot(
                x_pos,
                y_pos,
                marker=marker,
                markerfacecolor="white",
                markeredgecolor=color,
                markeredgewidth=2,
                markersize=18,
                alpha=0.95
            )

            plt.text(
                x_pos,
                y_pos,
                str(count),
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold",
                color=color
            )

    # ---------------- Legend ----------------
    legend_handles = []

    for label in unique_labels:

        class_name = label_encoder.inverse_transform(
            [int(label)]
        )[0]

        marker = markers[label % len(markers)]
        color = colors[label % len(colors)]

        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker=marker,
                linestyle="None",
                label=str(class_name),
                markerfacecolor="white",
                markeredgecolor=color,
                markeredgewidth=2,
                markersize=12
            )
        )

    plt.legend(
        handles=legend_handles,
        title="True Class Labels",
        loc="upper left",
        bbox_to_anchor=(1.22, 1.00),
        frameon=True
    )

    # ---------------- Titles and labels ----------------
    plt.title(
        f"U-Matrix with Class Count Markers\n"
        f"Model={model_name}, Grid={grid_size}, Sigma={sigma}, LR={lr}"
    )

    plt.xlabel("SOM X")
    plt.ylabel("SOM Y")

    filename = (
        f"{output_dir}/umatrix_class_count_markers_"
        f"{safe_filename(model_name)}_"
        f"grid{grid_size}_"
        f"sigma{sigma}_lr{lr}.png"
    )

    plt.tight_layout()

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


def plot_hit_map(som, data, model_name, sigma, lr, output_dir):
    """
    Hit map shows how many documents/news items are mapped to each neuron.
    High values mean many samples are represented by that neuron.
    """

    x_size = som._weights.shape[0]
    y_size = som._weights.shape[1]

    hit_map = np.zeros((x_size, y_size))

    for sample in data:
        winner = som.winner(sample)
        hit_map[winner] += 1

    plt.figure(figsize=(7, 6))

    plt.imshow(hit_map.T, cmap="Blues")
    plt.colorbar(label="Number of samples")

    for i in range(x_size):
        for j in range(y_size):
            plt.text(
                i,
                j,
                int(hit_map[i, j]),
                ha="center",
                va="center",
                fontsize=9
            )

    plt.title(
        f"SOM Hit Map\nModel={model_name}, Sigma={sigma}, LR={lr}"
    )

    plt.xlabel("SOM X")
    plt.ylabel("SOM Y")

    filename = (
        f"{output_dir}/hitmap_"
        f"{safe_filename(model_name)}_sigma{sigma}_lr{lr}.png"
    )

    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()


In [ ]:
# ============================================================
# PREPROCESSING
# ============================================================

def preprocess_tweet(text):

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-z\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# LOAD DATASET
# ============================================================

# Load dataset
def load_dataset(config):
    df = pd.read_parquet(config["dataset_path"])
    df["sentiment"] = df["rating"].map({
        1: "negative",
        2: "negative",
        3: "neutral",
        4: "neutral",
        5: "positive"
    })
    print("Original rows:", len(df))
    print("Original Category distribution:")
    print(df[config["label_column"]].value_counts())

    SAMPLES_PER_CLASS = config.get("samples_per_class", 100000)

    df_small = (
        df.groupby(config["label_column"], group_keys=False)
          .apply(lambda x: x.sample(
              n=min(len(x), SAMPLES_PER_CLASS),
              random_state=42
          ))
          .reset_index(drop=True)
    )

    print("\nReduced rows:", len(df_small))
    print("Reduced class distribution:")
    print(df_small[config["label_column"]].value_counts())

    texts = df_small[
        config["text_column"]
    ].astype(str).tolist()

    labels = df_small[
        config["label_column"]
    ].astype(str).tolist()

    texts = [
        preprocess_tweet(t)
        for t in texts
    ]

    return texts, labels


# ============================================================
# EMBEDDING MODELS
# ============================================================

TRANSFORMER_MODELS = {

    "MiniLM":
        "paraphrase-multilingual-MiniLM-L12-v2",

    "MPNet":
        "paraphrase-multilingual-mpnet-base-v2",

    "DistilBERT_STS":
        "distiluse-base-multilingual-cased-v2"
}


# ---------------- Word2Vec ----------------

def get_word2vec_embeddings(
    texts,
    vector_size=300,
    window=5,
    min_count=1,
    epochs=10
):

    tokenized = [
        t.split()
        for t in texts
    ]

    model = Word2Vec(
        sentences=tokenized,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        epochs=epochs
    )

    embeddings = []

    for tokens in tokenized:

        vectors = [
            model.wv[w]
            for w in tokens
            if w in model.wv
        ]

        if len(vectors) > 0:

            embeddings.append(
                np.mean(vectors, axis=0)
            )

        else:

            embeddings.append(
                np.zeros(vector_size)
            )

    return np.array(
        embeddings,
        dtype=np.float32
    )


# ---------------- Doc2Vec ----------------

def get_doc2vec_embeddings(
    texts,
    vector_size=300,
    window=5,
    min_count=1,
    epochs=20
):

    tagged_docs = [

        TaggedDocument(
            words=t.split(),
            tags=[i]
        )

        for i, t in enumerate(texts)
    ]

    model = Doc2Vec(
        documents=tagged_docs,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        epochs=epochs
    )

    embeddings = np.array(
        [
            model.dv[i]
            for i in range(len(texts))
        ],
        dtype=np.float32
    )

    return embeddings


# ---------------- Transformer Embeddings ----------------

def get_transformer_embeddings(
    texts,
    model_name
):

    model = SentenceTransformer(model_name)

    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=64
    )

    return embeddings


# ---------------- Unified Function ----------------

def compute_embeddings(
    texts,
    model_key
):

    if model_key == "Word2Vec_avg":

        return get_word2vec_embeddings(texts)

    elif model_key == "Doc2Vec":

        return get_doc2vec_embeddings(texts)

    elif model_key in TRANSFORMER_MODELS:

        return get_transformer_embeddings(
            texts,
            TRANSFORMER_MODELS[model_key]
        )

    else:

        raise ValueError(
            f"Unknown model: {model_key}"
        )


# ============================================================
# PCA
# ============================================================

def apply_pca(
    embeddings,
    n_components
):

    n_components = min(
        n_components,
        embeddings.shape[1] - 1
    )

    pca = PCA(
        n_components=n_components,
        random_state=42
    )

    reduced = pca.fit_transform(
        embeddings
    )

    return reduced


# ============================================================
# SOM
# ============================================================

def train_som(
    data,
    x,
    y,
    sigma,
    learning_rate,
    iterations,
    seed
):

    som = MiniSom(
        x=x,
        y=y,
        input_len=data.shape[1],
        sigma=sigma,
        learning_rate=learning_rate,
        random_seed=seed
    )

    som.random_weights_init(data)

    som.train_random(
        data,
        iterations
    )

    return som


def assign_clusters(
    som,
    data
):

    winners = np.array([
        som.winner(x)
        for x in data
    ])

    cluster_labels = np.array([

        w[0] * som._weights.shape[1] + w[1]

        for w in winners
    ])

    return cluster_labels


# ============================================================
# METRICS
# ============================================================

def purity_score(
    y_true,
    y_pred
):

    matrix = contingency_matrix(
        y_true,
        y_pred
    )

    return (
        np.sum(np.max(matrix, axis=0))
        / np.sum(matrix)
    )


def quantization_error(
    som,
    data
):

    return som.quantization_error(data)


def topographic_error(
    som,
    data
):

    errors = 0

    for x in data:

        winner = som.winner(x)

        activation = som.activate(x)

        flat = activation.flatten()

        sorted_idx = np.argsort(flat)

        second = np.unravel_index(
            sorted_idx[1],
            activation.shape
        )

        dist = (
            abs(winner[0] - second[0])
            +
            abs(winner[1] - second[1])
        )

        if dist > 1:
            errors += 1

    return errors / len(data)

In [ ]:
# ============================================================
# MAIN EXPERIMENT
# ============================================================

def run_experiments(config):

    # Load dataset
    texts, labels = load_dataset(config)
    output_dir = create_output_dir(
        config["results_file_name"].replace(".csv", "")
    )

    sample_size = len(texts)

    # Encode labels
    encoder = LabelEncoder()

    true_labels = encoder.fit_transform(labels)

    results = []

    # Loop over embedding models
    for model_name in config["embedding_models"]:

        print("\n" + "=" * 60)
        print(f"MODEL: {model_name}")
        print("=" * 60)

        # Compute embeddings
        embeddings = compute_embeddings(
            texts,
            model_name
        )

        # PCA
        if config["use_pca"]:

            embeddings = apply_pca(
                embeddings,
                config["pca_components"]
            )

        # SOM combinations
        for som_x, som_y in config["som_grids"]:

            for sigma in config["sigmas"]:

                for lr in config["learning_rates"]:

                    print(
                        f"SOM={som_x}x{som_y} | "
                        f"Sigma={sigma} | "
                        f"LR={lr}"
                    )

                    # Train SOM
                    som = train_som(
                        data=embeddings,
                        x=som_x,
                        y=som_y,
                        sigma=sigma,
                        learning_rate=lr,
                        iterations=config["som_iterations"],
                        seed=config["random_seed"]
                    )

                    # Cluster labels
                    cluster_labels = assign_clusters(
                        som,
                        embeddings
                    )

                    # Generate SOM visualizations
                    plot_umatrix_with_class_count_markers(
                        som=som,
                        data=embeddings,
                        true_labels=true_labels,
                        label_encoder=encoder,
                        model_name=model_name,
                        som_x=som_x,
                        som_y=som_y,
                        sigma=sigma,
                        lr=lr,
                        output_dir=output_dir
                    )

                    # Metrics
                    # silhouette = silhouette_score(
                    #     embeddings,
                    #     cluster_labels,
                    #     metric="cosine"
                    # )

                    silhouette = silhouette_score(
                        embeddings,
                        cluster_labels,
                        metric="cosine",
                        sample_size=5000,
                        random_state=42
                    )

                    ari = adjusted_rand_score(
                        true_labels,
                        cluster_labels
                    )

                    purity = purity_score(
                        true_labels,
                        cluster_labels
                    )

                    qe = quantization_error(
                        som,
                        embeddings
                    )

                    te = topographic_error(
                        som,
                        embeddings
                    )

                    # Store results
                    results.append({
                        "Model":
                            model_name,
                        "Samples Size":
                            sample_size,
                        "Grid Size":
                            f"{som_x}x{som_y}",
                        "Sigma":
                            sigma,
                        "Learning Rate":
                            lr,
                        "Iterations":
                            config["som_iterations"],
                        "Silhouette":
                            round(silhouette, 5),
                        "ARI":
                            round(ari, 5),
                        "Purity":
                            round(purity, 5),
                        "QE":
                            round(qe, 5),
                        "TE":
                            round(te, 5)
                    })

        # Memory cleanup
        del embeddings
        gc.collect()
    return pd.DataFrame(results)


# ============================================================
# RUN
# ============================================================

final_results = run_experiments(CONFIG)

Original rows: 1796
Original Category distribution:
sentiment
neutral     876
positive    611
negative    309
Name: count, dtype: int64

Reduced rows: 1796
Reduced class distribution:
sentiment
neutral     876
positive    611
negative    309
Name: count, dtype: int64

MODEL: Word2Vec_avg


/tmp/ipykernel_3426/1349794839.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


SOM=5x5 | Sigma=2.0 | LR=0.1

MODEL: Doc2Vec
SOM=5x5 | Sigma=2.0 | LR=0.1

MODEL: MiniLM


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

SOM=5x5 | Sigma=2.0 | LR=0.1


In [ ]:
# ============================================================
# SAVE CSV
# ============================================================

final_results.to_csv(
    CONFIG["results_file_name"],
    index=False
)

print("\n"+CONFIG["results_file_name"])

print("\nFinal Results:")
print(final_results)


Lithuanian_sentiment_analysis_dataset_results.csv

Final Results:
            Model  Samples Size Grid Size  Sigma  Learning Rate  Iterations  \
0    Word2Vec_avg          1796       5x5    2.0            0.1        1000   
1         Doc2Vec          1796       5x5    2.0            0.1        1000   
2          MiniLM          1796       5x5    2.0            0.1        1000   
3           MPNet          1796       5x5    2.0            0.1        1000   
4  DistilBERT_STS          1796       5x5    2.0            0.1        1000   

   Silhouette      ARI   Purity       QE       TE  
0    -0.32534  0.00529  0.48942  0.10502  0.75278  
1    -0.21719  0.01043  0.50947  0.20257  0.57461  
2     0.04079  0.01027  0.52116  2.15892  0.17038  
3     0.04731  0.01369  0.53396  1.41472  0.09243  
4     0.02926  0.00906  0.51002  0.51489  0.16036  


In [ ]:
best_result = final_results.sort_values(
    "ARI",
    ascending=False
).iloc[0]

print("\nBest Configuration:")
print(best_result)


Best Configuration:
Model              MPNet
Samples Size        1796
Grid Size            5x5
Sigma                2.0
Learning Rate        0.1
Iterations          1000
Silhouette       0.04731
ARI              0.01369
Purity           0.53396
QE               1.41472
TE               0.09243
Name: 3, dtype: object


In [ ]:
import os
import shutil

if CONFIG.get("auto_download_results", False):
    from google.colab import files

    # ---------------- Download results CSV ----------------
    results_file = CONFIG["results_file_name"]

    if os.path.exists(results_file):
        files.download(results_file)
        print("Downloaded results file:", results_file)
    else:
        print("Results file not found:", results_file)

    # ---------------- Download SOM plots folder ----------------
    plots_folder = "som_plots_" + CONFIG["results_file_name"].replace(".csv", "")

    if os.path.exists(plots_folder):
        zip_file = plots_folder + ".zip"

        shutil.make_archive(
            plots_folder,
            "zip",
            plots_folder
        )

        files.download(zip_file)
        print("Downloaded plots ZIP:", zip_file)
    else:
        print("Plots folder not found:", plots_folder)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded results file: Lithuanian_sentiment_analysis_dataset_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded plots ZIP: som_plots_Lithuanian_sentiment_analysis_dataset_results.zip
